# 00 — Environment & Colab

**Data & AI Engineering Lab**

> Learn data engineering by building, measuring, breaking and designing systems.

### Learning objectives

By the end of this notebook you will be able to:

- verify the Python/Colab runtime;
- install and inspect dependencies reproducibly;
- distinguish notebook state from durable project files;
- generate a deterministic dataset;
- run a small correctness check;
- understand the **CORE-COLAB** contract used throughout this course.

### The engineering habit

A notebook is not production code just because it runs once.

For this course, every core notebook should aim to be:

**Fresh runtime → setup → deterministic input → experiment → verification → useful result.**


## Capability contract

**Capability:** C01 — System Foundations  
**Workstream:** Foundation  
**Primary roles:** Data Engineer|Analytics Engineer  
**You will prove:** Explain the mechanism and reproduce it with a deterministic experiment.


## 🖊️ Sketch note

![Sketch note — Environment & Colab](../assets/sketch-notes/00-environment-colab.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

This foundation notebook does not have one dedicated Acme Commerce ticket. It is a **cross-cutting prerequisite** used throughout the apprenticeship. Use it whenever a scenario requires the underlying runtime, Python or basic data-handling skill.

**Scenario map:** see [APPRENTICESHIP_MAP.md](../APPRENTICESHIP_MAP.md).


## Why this exists

Start with the engineering problem. Know what would go wrong if this mechanism did not exist.


## Predict before you run

Before executing the environment checks, predict which properties should remain true after a fresh restart.


## Measure

Record the Python version, dependency version and dataset invariants.


## Diagnose

If a check fails, identify whether the cause is runtime, dependency, state or data.


## Production bridge

A production environment replaces notebook convenience with pinned dependencies, repeatable builds, logging and explicit configuration.


In [ ]:
# Core setup — intentionally small and fast.
from pathlib import Path
import sys, platform, json, os

REPO_ROOT = Path.cwd()
COMMON = REPO_ROOT / "common"

# When opened directly from GitHub in Colab, the notebook's cwd is normally
# the repository root only if the user has cloned it. The next cell handles
# the common alternative: notebook-local execution.
if not COMMON.exists():
    COMMON = Path("/content") / "data-ai-engineering-lab" / "common"

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())
print("Common helpers:", COMMON.resolve() if COMMON.exists() else "not found")


In [ ]:
# Install only what we need for this first notebook.
# In later notebooks, dependency installation will be explicit and minimal.
%pip -q install "pandas>=2,<4"


In [ ]:
# Verify the environment.
import pandas as pd

print("pandas:", pd.__version__)
assert tuple(map(int, pd.__version__.split(".")[:1])) >= (2,), "Expected pandas >= 2"
print("Environment check: PASS")


## 1. Deterministic data is a superpower

We will generate the same dataset every time.

That makes experiments:

- reproducible;
- debuggable;
- comparable;
- easy to test.

Randomness is useful, but it should be controlled when it affects a lesson.


In [ ]:
import random
from datetime import date, timedelta

SEED = 42
rng = random.Random(SEED)

rows = []
start = date(2026, 1, 1)

for i in range(20):
    rows.append({
        "event_id": i + 1,
        "customer_id": rng.randint(100, 105),
        "event_date": start + timedelta(days=rng.randint(0, 9)),
        "amount": round(rng.uniform(10, 250), 2),
    })

df = pd.DataFrame(rows)
df.head()


In [ ]:
# A tiny correctness check.
assert len(df) == 20
assert df["event_id"].is_unique
assert df["amount"].ge(0).all()

print("Rows:", len(df))
print("Unique customers:", df["customer_id"].nunique())
print("Correctness checks: PASS")


## 2. What can go wrong?

Try changing `SEED = 42` to another value.

Then ask:

1. Which outputs changed?
2. Which properties should remain true?
3. Which assertions still protect us?

This is the beginning of **invariant-based testing**: instead of checking one exact output, we check properties that must always hold.


In [ ]:
# Exercise
# Change SEED above, rerun the data-generation cell, and verify that:
# - there are still exactly 20 rows
# - event_id remains unique
# - amount remains non-negative
#
# Then write one additional assertion below.

assert df["customer_id"].between(100, 105).all()
print("Your added invariant: PASS")


## 3. Colab contract

For this project, notebook status will be labelled:

- 🟢 **CORE-COLAB** — works entirely inside Colab.
- 🟡 **COLAB-WITH-LIGHT-INFRA** — starts a local/subprocess service.
- 🟠 **COLAB-SIMULATION** — simulates distributed/production behaviour.
- 🔵 **OPTIONAL-PRODUCTION** — external infrastructure; never required for the core path.

We will not pretend that a temporary Colab VM is equivalent to a production cluster.

Instead, we will use Colab to learn the *principles* and explicitly discuss what changes at production scale.


## Engineering challenge

Add three checks that would make this dataset safer for a downstream pipeline.

Examples:

- no missing customer IDs;
- dates are within the expected lesson range;
- amounts have a reasonable upper bound.

**Next:** we move from runtime setup to real data files, JSON, and APIs.
